In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/results"
table_name = f"{catalog_name}.{bronze_schema}.results"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

# Define schema for results JSON files with StructType and StructField
results_schema = StructType([
    StructField("constructorId", StringType(), True),
    StructField("date", DateType(), True),
    StructField("driverId", StringType(), True),
    StructField("grid", IntegerType(), True),
    StructField("laps", IntegerType(), True),
    StructField("number", IntegerType(), True),
    StructField("points", DoubleType(), True),
    StructField("position", IntegerType(), True),
    StructField("positionText", StringType(), True),
    StructField("raceName", StringType(), True),
    StructField("round", IntegerType(), True),
    StructField("season", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("url", StringType(), True)
])

# Read results JSON files from volume with defined schema (multiple files)
results_df = (spark.read
    .schema(results_schema)
    .option("mode", "FAILFAST")
    .json(source_file)
)

#display(results_df)

In [0]:
# Add ingestion timestamp and source file path metadata using helper function
results_with_metadata_df = add_ingestion_metadata(results_df)

display(results_with_metadata_df)

In [0]:
# Write results data to bronze layer table
(results_with_metadata_df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(table_name)
)

print(f"✅ Results data successfully written to {table_name} table")

In [0]:
%sql
SELECT 
  season,count(*)
FROM formula1.bronze.results
group by season order by season